In [18]:
# ===============================
# 1. IMPORT
# ===============================
import pandas as pd
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline

from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier

# ===============================
# 2. LOAD DATA
# ===============================
df = pd.read_csv("online_shoppers_intention.csv")

# ===============================
# 3. PREPROCESSING
# ===============================
# Target
y = df["Revenue"]

# Feature
X = df.drop("Revenue", axis=1)

# One-hot encoding (categorical → numeric)
X = pd.get_dummies(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling khusus MLP
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ===============================
# 4. MLP MODEL
# ===============================
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=300,
    random_state=42
)

start = time.time()
mlp.fit(X_train_scaled, y_train)
rt_train_mlp = time.time() - start

start = time.time()
pred_mlp = mlp.predict(X_test_scaled)
rt_pred_mlp = time.time() - start

acc_mlp = accuracy_score(y_test, pred_mlp)

print(f"MLP Accuracy: {acc_mlp:.4f}")
print(f"Runtime train: {rt_train_mlp:.2f}s | predict: {rt_pred_mlp:.2f}s\n")

# ===============================
# 5. RANDOM FOREST MODEL
# ===============================
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

start = time.time()
rf.fit(X_train, y_train)
rt_train_rf = time.time() - start

start = time.time()
pred_rf = rf.predict(X_test)
rt_pred_rf = time.time() - start

acc_rf = accuracy_score(y_test, pred_rf)

print(f"Random Forest Accuracy: {acc_rf:.4f}")
print(f"Runtime train: {rt_train_rf:.2f}s | predict: {rt_pred_rf:.2f}s\n")

# ===============================
# 6. COMPARISON TABLE
# ===============================
results = pd.DataFrame({
    "Model": ["MLP", "Random Forest"],
    "Accuracy": [acc_mlp, acc_rf],
    "Runtime Training (s)": [rt_train_mlp, rt_train_rf],
    "Runtime Prediction (s)": [rt_pred_mlp, rt_pred_rf]
})

# format persen
results["Accuracy"] = results["Accuracy"].apply(lambda x: f"{x*100:.2f}%")

print("=== MODEL COMPARISON ===")
print(results.to_string(index=False))

/Users/mac/sc_env/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
Python(94008) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


MLP Accuracy: 0.8642
Runtime train: 4.86s | predict: 0.00s

Random Forest Accuracy: 0.8978
Runtime train: 0.47s | predict: 0.01s

=== MODEL COMPARISON ===
        Model Accuracy  Runtime Training (s)  Runtime Prediction (s)
          MLP   86.42%              4.856007                0.002172
Random Forest   89.78%              0.466087                0.014867


In [21]:
# =============================================================
# TUGAS: PERBANDINGAN MLP (NEURAL NETWORK) VS XGBOOST (PRO TREE)
# Dataset: Online Shoppers Purchasing Intention
# =============================================================

import pandas as pd
import time
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier # Pastikan sudah: pip install xgboost

# 1. LOAD DATA
df = pd.read_csv("online_shoppers_intention.csv")

# 2. PREPROCESSING
y = df["Revenue"].astype(int)
X = df.drop("Revenue", axis=1)

# One-hot encoding untuk data kategorikal
X = pd.get_dummies(X)

# Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling (Wajib untuk MLP agar akurasi tidak anjlok)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("--- Memulai Training Model (Optimized) ---\n")

# =============================================================
# 3. MODEL 1: MULTILAYER PERCEPTRON (MLP) - SETTING PRO
# =============================================================
# Kita buat arsitektur 3 layer tersembunyi agar lebih "dalam"
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32), 
    activation='relu',
    solver='adam',
    max_iter=500,         # Memberi waktu belajar lebih lama
    alpha=0.0001,         # Regularisasi L2 untuk cegah overfitting
    learning_rate_init=0.001,
    random_state=42,
    early_stopping=True,  # Berhenti otomatis jika akurasi sudah mentok
    validation_fraction=0.1
)

start_t = time.time()
mlp.fit(X_train_scaled, y_train)
train_time_mlp = time.time() - start_t

start_p = time.time()
pred_mlp = mlp.predict(X_test_scaled)
pred_time_mlp = time.time() - start_p

acc_mlp = accuracy_score(y_test, pred_mlp)

# =============================================================
# 4. MODEL 2: XGBOOST (PRO BOOSTING TREE) - SETTING PRO
# =============================================================
# Menggunakan learning rate kecil dan pohon yang banyak agar detail
xgb = XGBClassifier(
    n_estimators=1000,    # Jumlah pohon banyak agar akurasi tajam
    learning_rate=0.01,   # Belajar pelan tapi pasti
    max_depth=6,          # Kedalaman pohon yang optimal
    subsample=0.8,        # Menggunakan 80% data acak per iterasi
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

start_t = time.time()
xgb.fit(X_train, y_train) # XGBoost tidak wajib pakai data scaled
train_time_xgb = time.time() - start_t

start_p = time.time()
pred_xgb = xgb.predict(X_test)
pred_time_xgb = time.time() - start_p

acc_xgb = accuracy_score(y_test, pred_xgb)

# =============================================================
# 5. OUTPUT HASIL AKHIR
# =============================================================
print("=========================================")
print("          HASIL PERBANDINGAN             ")
print("=========================================")
print(f"MLP (Neural Network) Accuracy : {acc_mlp*100:.2f}%")
print(f"XGBoost (Boosting Tree) Acc   : {acc_xgb*100:.2f}%")
print("-----------------------------------------")
print(f"Training Time MLP: {train_time_mlp:.2f}s | XGB: {train_time_xgb:.2f}s")
print(f"Predict Time  MLP: {pred_time_mlp:.4f}s | XGB: {pred_time_xgb:.4f}s")
print("=========================================")

# Tips: Copy hasil di bawah ini untuk tabel Laporan
comparison_df = pd.DataFrame({
    "Metrik": ["Akurasi", "Waktu Latih (s)", "Waktu Prediksi (s)"],
    "MLP (Neural Network)": [f"{acc_mlp*100:.2f}%", round(train_time_mlp,2), round(pred_time_mlp,4)],
    "XGBoost (Boosting)": [f"{acc_xgb*100:.2f}%", round(train_time_xgb,2), round(pred_time_xgb,4)]
})
print("\nTABEL UNTUK LAPORAN:")
print(comparison_df.to_string(index=False))

--- Memulai Training Model (Optimized) ---



/Users/mac/sc_env/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [05:05:12] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


          HASIL PERBANDINGAN             
MLP (Neural Network) Accuracy : 89.01%
XGBoost (Boosting Tree) Acc   : 90.15%
-----------------------------------------
Training Time MLP: 0.50s | XGB: 1.31s
Predict Time  MLP: 0.0012s | XGB: 0.0083s

TABEL UNTUK LAPORAN:
            Metrik MLP (Neural Network) XGBoost (Boosting)
           Akurasi               89.01%             90.15%
   Waktu Latih (s)                  0.5               1.31
Waktu Prediksi (s)               0.0012             0.0083


In [23]:
# ===============================
# 1. IMPORT
# ===============================
import pandas as pd
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

# ===============================
# 2. LOAD DATA
# ===============================
df = pd.read_csv("online_shoppers_intention.csv")

# ===============================
# 3. PREPROCESSING
# ===============================
y = df["Revenue"]
X = df.drop("Revenue", axis=1)

# One-hot encoding
X = pd.get_dummies(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling (khusus MLP)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ===============================
# 4. MLP (TUNED)
# ===============================
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    alpha=0.0005,
    learning_rate='adaptive',
    max_iter=500,
    early_stopping=True,
    random_state=42
)

start = time.time()
mlp.fit(X_train_scaled, y_train)
rt_train_mlp = time.time() - start

start = time.time()
pred_mlp = mlp.predict(X_test_scaled)
rt_pred_mlp = time.time() - start

acc_mlp = accuracy_score(y_test, pred_mlp)

print(f"MLP Accuracy: {acc_mlp:.4f}")
print(f"Runtime train: {rt_train_mlp:.2f}s | predict: {rt_pred_mlp:.4f}s\n")

# ===============================
# 5. RANDOM FOREST (TUNED)
# ===============================
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

start = time.time()
rf.fit(X_train, y_train)
rt_train_rf = time.time() - start

start = time.time()
pred_rf = rf.predict(X_test)
rt_pred_rf = time.time() - start

acc_rf = accuracy_score(y_test, pred_rf)

print(f"Random Forest Accuracy: {acc_rf:.4f}")
print(f"Runtime train: {rt_train_rf:.2f}s | predict: {rt_pred_rf:.4f}s\n")

# ===============================
# 6. LIGHTGBM (HIGH PERFORMANCE)
# ===============================
lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

start = time.time()
lgbm.fit(X_train, y_train)
rt_train_lgbm = time.time() - start

start = time.time()
pred_lgbm = lgbm.predict(X_test)
rt_pred_lgbm = time.time() - start

acc_lgbm = accuracy_score(y_test, pred_lgbm)

print(f"LightGBM Accuracy: {acc_lgbm:.4f}")
print(f"Runtime train: {rt_train_lgbm:.2f}s | predict: {rt_pred_lgbm:.4f}s\n")

# ===============================
# 7. COMPARISON TABLE
# ===============================
results = pd.DataFrame({
    "Model": ["MLP", "Random Forest", "LightGBM"],
    "Accuracy": [acc_mlp, acc_rf, acc_lgbm],
    "Runtime Training (s)": [rt_train_mlp, rt_train_rf, rt_train_lgbm],
    "Runtime Prediction (s)": [rt_pred_mlp, rt_pred_rf, rt_pred_lgbm]
})

# Format persen
results["Accuracy"] = results["Accuracy"].apply(lambda x: f"{x*100:.2f}%")

print("\n=== MODEL COMPARISON ===")
print(results.to_string(index=False))

MLP Accuracy: 0.8942
Runtime train: 0.83s | predict: 0.0050s

Random Forest Accuracy: 0.9006
Runtime train: 0.53s | predict: 0.0266s

[LightGBM] [Info] Number of positive: 1526, number of negative: 8338
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000780 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1904
[LightGBM] [Info] Number of data points in the train set: 9864, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.154704 -> initscore=-1.698173
[LightGBM] [Info] Start training from score -1.698173


Python(97141) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


LightGBM Accuracy: 0.8990
Runtime train: 1.88s | predict: 0.0207s


=== MODEL COMPARISON ===
        Model Accuracy  Runtime Training (s)  Runtime Prediction (s)
          MLP   89.42%              0.831745                0.005006
Random Forest   90.06%              0.532553                0.026588
     LightGBM   89.90%              1.879262                0.020749


In [24]:
import pandas as pd
import time
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE # Rahasia akurasi mantap

# 1. LOAD DATA
df = pd.read_csv("online_shoppers_intention.csv")

# 2. PREPROCESSING
y = df["Revenue"].astype(int)
X = pd.get_dummies(df.drop("Revenue", axis=1))

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# HANDLING IMBALANCED DATA (SMOTE)
# Ini akan membuat model lebih pintar mengenali pembeli
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# 3. DEFINISI PIPELINE (Agar rapi & profesional)
# Pipeline memastikan scaling hanya dilakukan pada data yang tepat
def create_pipeline(model, use_scaler=False):
    if use_scaler:
        return Pipeline([
            ('scaler', StandardScaler()),
            ('clf', model)
        ])
    else:
        return Pipeline([
            ('clf', model)
        ])

# 4. MODEL DEFINITIONS (TUNED)
models = {
    "MLP (Neural Network)": create_pipeline(MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        activation='relu', solver='adam', max_iter=500,
        early_stopping=True, random_state=42
    ), use_scaler=True),
    
    "Random Forest": create_pipeline(RandomForestClassifier(
        n_estimators=500, max_depth=20, random_state=42, n_jobs=-1
    )),
    
    "LightGBM": create_pipeline(LGBMClassifier(
        n_estimators=1000, learning_rate=0.01, num_leaves=64, 
        boosting_type='gbdt', random_state=42, verbosity=-1
    ))
}

# 5. EXECUTION & EVALUATION
final_results = []

print("--- Memulai Eksperimen High Performance ---")

for name, pipe in models.items():
    # Training
    start_t = time.time()
    pipe.fit(X_train_res, y_train_res) # Pakai data hasil SMOTE
    train_time = time.time() - start_t
    
    # Prediction
    start_p = time.time()
    preds = pipe.predict(X_test)
    pred_time = time.time() - start_p
    
    acc = accuracy_score(y_test, preds)
    
    final_results.append({
        "Model": name,
        "Accuracy": f"{acc*100:.2f}%",
        "Train Time (s)": round(train_time, 3),
        "Predict Time (s)": round(pred_time, 5)
    })
    print(f"Selesai: {name}")

# 6. DISPLAY TABLE
results_df = pd.DataFrame(final_results)
print("\n=== TABEL PERBANDINGAN FINAL ===")
print(results_df.to_string(index=False))

--- Memulai Eksperimen High Performance ---
Selesai: MLP (Neural Network)
Selesai: Random Forest
Selesai: LightGBM

=== TABEL PERBANDINGAN FINAL ===
               Model Accuracy  Train Time (s)  Predict Time (s)
MLP (Neural Network)   87.75%           1.555           0.00277
       Random Forest   88.93%           1.285           0.05156
            LightGBM   88.89%           7.190           0.04448


In [1]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE

# 1. LOAD DATA
# Pastikan file CSV ada di folder yang sama dengan file .py ini
df = pd.read_csv("online_shoppers_intention.csv")

# 2. PREPROCESSING
# Ubah kolom target menjadi angka (0 dan 1)
y = df["Revenue"].astype(int)
# One-hot encoding untuk kolom kategori (Month, VisitorType, dll)
X = pd.get_dummies(df.drop("Revenue", axis=1))

# Split data (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Menangani ketidakseimbangan data (SMOTE) agar model tidak berat sebelah
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Scaling (Sangat WAJIB untuk TensorFlow agar sarafnya tidak kaget dengan angka besar)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

print(f"Jumlah Fitur setelah Encoding: {X_train_scaled.shape[1]}")

# ==========================================
# 3. MODEL 1: NEURAL NETWORK (TENSORFLOW)
# ==========================================
print("\n--- Training Model 1: TensorFlow (Neural Network) ---")

model_nn = Sequential([
    Dense(128, activation='relu', input_dim=X_train_scaled.shape[1]),
    Dropout(0.3), # Mencegah menghafal (overfitting)
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid') # Sigmoid karena outputnya biner (Beli/Tidak)
])

model_nn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Berhenti otomatis jika akurasi sudah maksimal (biar cepat)
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

start_t = time.time()
model_nn.fit(X_train_scaled, y_train_res, epochs=50, batch_size=32, 
             validation_split=0.2, callbacks=[early_stop], verbose=1)
train_time_nn = time.time() - start_t

# Prediksi NN
y_pred_nn_prob = model_nn.predict(X_test_scaled)
y_pred_nn = (y_pred_nn_prob > 0.5).astype(int)
acc_nn = accuracy_score(y_test, y_pred_nn)

# ==========================================
# 4. MODEL 2: RANDOM FOREST (STANDAR)
# ==========================================
print("\n--- Training Model 2: Random Forest (Pembanding) ---")

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

start_t = time.time()
rf_model.fit(X_train_res, y_train_res)
train_time_rf = time.time() - start_t

# Prediksi RF
y_pred_rf = rf_model.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

# ==========================================
# 5. HASIL AKHIR UNTUK LAPORAN
# ==========================================
print("\n" + "="*40)
print("HASIL PERBANDINGAN METODE")
print("="*40)
print(f"{'Metode':<25} | {'Akurasi':<10} | {'Waktu Latih':<10}")
print("-" * 45)
print(f"{'Neural Network (TF)':<25} | {acc_nn*100:>8.2f}% | {train_time_nn:>8.2f}s")
print(f"{'Random Forest (Std)':<25} | {acc_rf*100:>8.2f}% | {train_time_rf:>8.2f}s")
print("="*40)

FileNotFoundError: [Errno 2] No such file or directory: 'online_shoppers_intention.csv'